# ENSO Forecasting: CART Decision Tree Regression

Single decision-tree baseline for next-month Nino 3.4 SST prediction from 12 lagged monthly values.

The same chronological split and regression metrics are used for comparison.


## 1. Import Libraries

In [ ]:
# AI-assisted: Initial structure and implementation were drafted with Codex.
# Human edits: Adapted for ENSO dataset, selected features/targets, verified outputs, and tuned evaluation.

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

## 2. Load the Data

In [ ]:
# Locate the data folder from either the project root or notebooks folder.
data_dir = Path("../data")
if not data_dir.exists():
    data_dir = Path("data")

X_train = pd.read_csv(data_dir / "X_train.csv")
X_test = pd.read_csv(data_dir / "X_test.csv")
y_train = pd.read_csv(data_dir / "y_train.csv").to_numpy(dtype=np.float32).ravel()
y_test = pd.read_csv(data_dir / "y_test.csv").to_numpy(dtype=np.float32).ravel()

print("Loaded data from:", data_dir.resolve())
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 3. Prepare Data for Model

In [ ]:
total_samples = len(X_train) + len(X_test)
train_pct = len(X_train) / total_samples * 100
test_pct = len(X_test) / total_samples * 100

print(f"Total samples: {total_samples}")
print(f"Train samples: {len(X_train)} ({train_pct:.1f}%)")
print(f"Test samples: {len(X_test)} ({test_pct:.1f}%)")
print(f"Features: {list(X_train.columns)}")
print("Target: target_t")
print("Split method: chronological, no shuffle")

## 4. Build the Model

In [ ]:
model = DecisionTreeRegressor(
    min_samples_split=20,
    random_state=42
)

model

## 5. Train the Model

In [ ]:
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

## 6. Evaluate on the Test Set

In [ ]:
def evaluate_regression(y_true, y_pred, split_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"--- {split_name} ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R^2:  {r2:.4f}")

    return rmse, mae, r2

print("CART Decision Tree Performance")
train_metrics = evaluate_regression(y_train, y_pred_train, "Train")
test_metrics = evaluate_regression(y_test, y_pred_test, "Test")

## 7. Visualize Results

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test, label="Actual", linewidth=2)
plt.plot(y_pred_test, label="CART Predicted", linestyle="--", linewidth=2)
plt.xlabel("Test Sample")
plt.ylabel("Scaled SST")
plt.title("CART Predictions vs Actual Test Values")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(data_dir / "cart_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
feature_importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print(feature_importance_df.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.barh(feature_importance_df["Feature"], feature_importance_df["Importance"], color="steelblue")
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title("CART Feature Importances")
plt.tight_layout()
plt.savefig(data_dir / "cart_feature_importances.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save Test Predictions and Metrics

In [ ]:
predictions_df = pd.DataFrame({
    "y_actual": y_test,
    "y_pred_CART": y_pred_test
})
predictions_df.to_csv(data_dir / "cart_test_predictions.csv", index=False)

metrics_df = pd.DataFrame({
    "Model": ["CART Decision Tree"],
    "RMSE_Train": [train_metrics[0]],
    "MAE_Train": [train_metrics[1]],
    "R2_Train": [train_metrics[2]],
    "RMSE_Test": [test_metrics[0]],
    "MAE_Test": [test_metrics[1]],
    "R2_Test": [test_metrics[2]],
})
metrics_df.to_csv(data_dir / "cart_metrics.csv", index=False)

display(metrics_df)
print("Saved predictions to:", (data_dir / "cart_test_predictions.csv").resolve())
print("Saved metrics to:", (data_dir / "cart_metrics.csv").resolve())